<a href="https://colab.research.google.com/github/pablouriellopez68-sketch/tecnm-lenguajes-automatas-2/blob/main/laboratorios/sesion15/sesion15_laboratorio_pablo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lenguajes y Autómatas II (SCD-1016)
## Sesión 15 · Bloques Anidados y Ciclo de Vida de Identificadores

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Qué pasa con las variables de un bloque cuando el programa cierra la llave y sale de él?
>
> **Objetivo de la sesión:** Implementar push y pop de ámbitos enlazados, comprobando que al cerrar un bloque el cursor vuelve al padre y las variables del bloque dejan de ser visibles.

**Entregable:** tus funciones push_ambito() y pop_ambito() en artesano_sesion15_c.c, la salida del harness y la bitácora en auditorias/auditoria_sesion15.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Recibes el ámbito activo y un nombre; push_ambito() reserva un ámbito nuevo enlazado al actual y pop_ambito() libera el actual y devuelve el padre. El harness revisa nombres, profundidad y que pop sobre la raíz no destruya el programa.

Al final de la sesión tendrás: **Dos operaciones push y pop que construyen y deshacen la cadena de ámbitos sin perder el padre.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 15](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion15.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion15.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion15.h"

Ambito *push_ambito(Ambito *actual, const char *nombre);
Ambito *pop_ambito(Ambito *actual);
```

- push_ambito() devuelve NULL si el ámbito actual o el nombre son NULL.
- pop_ambito() devuelve NULL sobre la raíz y no libera memoria que no le pertenece.
- Cada push se empareja con un pop: no se dejan ámbitos colgando en la cadena.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion15_c.c` junto con `artesano_sesion15_c.c`
> (o `ia_sesion15_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `nombre del ambito nuevo` | `if` |
| 2 | `profundidad con dos push` | 3 |
| 3 | `pop devuelve el padre` | 1 |
| 4 | `pop sobre la raiz` | 1 |
| 5 | `push con padre NULL` | 1 |
| 6 | `profundidad tras pop` | 2 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 15 - Push y pop de ambitos ===
Caso                               esperado:              obtenido:              resultado
[ 1] nombre del ambito nuevo            esperado: if           obtenido: if           OK
[ 2] profundidad con dos push           esperado: 3            obtenido: 3            OK
[ 3] pop devuelve el padre              esperado: 1            obtenido: 1            OK
[ 4] pop sobre la raiz                  esperado: 1            obtenido: 1            OK
[ 5] push con padre NULL                esperado: 1            obtenido: 1            OK
[ 6] profundidad tras pop               esperado: 2            obtenido: 2            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion15.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] push_ambito() enlaza el nuevo ámbito con el padre y conserva el nombre recibido.
- [ ] pop_ambito() devuelve exactamente el padre y libera solo el ámbito que recibe.
- [ ] La profundidad de la cadena sube con cada push y baja con cada pop.


## Paso 0 · Entorno


In [8]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion15_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [9]:
%%writefile api_sesion15.h
/* api_sesion15.h - contrato de la Sesion 15
 * Push y pop de ámbitos
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION15_H
#define API_SESION15_H

typedef struct Ambito {
    const char *nombre;
    struct Ambito *padre;
} Ambito;

Ambito *push_ambito(Ambito *actual, const char *nombre);
Ambito *pop_ambito(Ambito *actual);

#endif /* API_SESION15_H */


Overwriting api_sesion15.h


In [10]:
%%writefile artesano_sesion15_c.c
/* artesano_sesion15_c.c - Fase 1: El Artesano (sin IA)
 * Push y pop de ámbitos.
 *
 * Implementa lo que declara api_sesion15.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion15.h"
#include <stdio.h>
#include <string.h>
#include <stdlib.h>

Ambito *push_ambito(Ambito *actual, const char *nombre) {
    if (actual == NULL || nombre == NULL) return NULL;
    Ambito *nuevo = malloc(sizeof *nuevo);
    if (nuevo == NULL) return NULL;
    nuevo->nombre = nombre;
    nuevo->padre = actual;
    return nuevo;
}

Ambito *pop_ambito(Ambito *actual) {
    if (actual == NULL || actual->padre == NULL) return NULL;
    Ambito *padre = actual->padre;
    free(actual);
    return padre;
}


Overwriting artesano_sesion15_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion15_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [11]:
%%writefile ia_sesion15_c.c
/* ia_sesion15_c.c - Fase 2: El Cientifico
 *
 * Codigo devuelto por la IA, con la MISMA firma que declara
 * api_sesion15.h. No trae main().
 */
#include "api_sesion15.h"
#include <stdlib.h>

Ambito *push_ambito(Ambito *actual, const char *nombre) {
    if (actual == NULL || nombre == NULL) return NULL;
    Ambito *nuevo = malloc(sizeof *nuevo);
    if (nuevo == NULL) return NULL;
    nuevo->nombre = nombre;
    nuevo->padre = actual;
    return nuevo;
}

Ambito *pop_ambito(Ambito *actual) {
    if (actual == NULL || actual->padre == NULL) return NULL;
    Ambito *padre = actual->padre;
    free(actual);
    return padre;
}


Overwriting ia_sesion15_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [12]:
%%writefile harness_sesion15_c.c
/* harness_sesion15_c.c - auditoria automatica de la Sesion 15
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion15_c.c artesano_sesion15_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion15.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 15 - Push y pop de ambitos ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { Ambito raiz = {"global", NULL}; Ambito *a = push_ambito(&raiz, "if"); const char *r = a ? a->nombre : "(nulo)"; pop_ambito(a); verificar_str("nombre del ambito nuevo", "if", r); }
    { Ambito raiz = {"global", NULL}; Ambito *a = push_ambito(&raiz, "if"); Ambito *b = push_ambito(a, "while"); int r = 0; for (Ambito *p = b; p != NULL; p = p->padre) r++; pop_ambito(b); pop_ambito(a); verificar_int("profundidad con dos push", (long)(3), (long)(r)); }
    { Ambito raiz = {"global", NULL}; Ambito *a = push_ambito(&raiz, "if"); int r = (pop_ambito(a) == &raiz); verificar_int("pop devuelve el padre", (long)(1), (long)(r)); }
    { Ambito raiz = {"global", NULL}; int r = (pop_ambito(&raiz) == NULL); verificar_int("pop sobre la raiz", (long)(1), (long)(r)); }
    { int r = (push_ambito(NULL, "x") == NULL); verificar_int("push con padre NULL", (long)(1), (long)(r)); }
    { Ambito raiz = {"global", NULL}; Ambito *a = push_ambito(&raiz, "if"); Ambito *b = push_ambito(a, "while"); Ambito *c = pop_ambito(b); int r = 0; for (Ambito *p = c; p != NULL; p = p->padre) r++; pop_ambito(c); verificar_int("profundidad tras pop", (long)(2), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion15.md" "\n");
    return 0;
}


Overwriting harness_sesion15_c.c


In [13]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion15_c.c artesano_sesion15_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 15 - Push y pop de ambitos ===
Caso                               esperado:              obtenido:              resultado
[ 1] nombre del ambito nuevo            esperado: if           obtenido: if           OK
[ 2] profundidad con dos push           esperado: 3            obtenido: 3            OK
[ 3] pop devuelve el padre              esperado: 1            obtenido: 1            OK
[ 4] pop sobre la raiz                  esperado: 1            obtenido: 1            OK
[ 5] push con padre NULL                esperado: 1            obtenido: 1            OK
[ 6] profundidad tras pop               esperado: 2            obtenido: 2            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion15.md


In [14]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion15_c.c ia_sesion15_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 15 - Push y pop de ambitos ===
Caso                               esperado:              obtenido:              resultado
[ 1] nombre del ambito nuevo            esperado: if           obtenido: if           OK
[ 2] profundidad con dos push           esperado: 3            obtenido: 3            OK
[ 3] pop devuelve el padre              esperado: 1            obtenido: 1            OK
[ 4] pop sobre la raiz                  esperado: 1            obtenido: 1            OK
[ 5] push con padre NULL                esperado: 1            obtenido: 1            OK
[ 6] profundidad tras pop               esperado: 2            obtenido: 2            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion15.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion15.md` a `auditorias/auditoria_sesion15.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
